<a href="https://colab.research.google.com/github/MaayanSal20/BlueBerry/blob/main/train_blueberry_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MODEL

In [1]:
!pip install tensorflow scikit-learn matplotlib pillow -q

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
import zipfile
import shutil

BASE_DIR = Path("/content/drive/MyDrive/BlueberryDiseaseProject")
ZIP_PATH = BASE_DIR / "dataset" / "blueberry_dataset.zip"
DATASET_DIR = Path("/content/blueberry_dataset")

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall("/content")

print("✅ Dataset extracted")

for folder in DATASET_DIR.iterdir():
    if folder.is_dir():
        print(folder.name, ":", len(list(folder.glob("*"))), "images")

✅ Dataset extracted
healthy : 30 images
leaf_spot : 16 images
mummy_berry : 21 images
botrytis : 26 images
anthracnose : 23 images


In [3]:
from pathlib import Path
from PIL import Image
import shutil

CLEAN_DIR = Path("/content/blueberry_dataset_clean")

if CLEAN_DIR.exists():
    shutil.rmtree(CLEAN_DIR)

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}

for class_dir in DATASET_DIR.iterdir():
    if not class_dir.is_dir():
        continue

    out_class_dir = CLEAN_DIR / class_dir.name
    out_class_dir.mkdir(parents=True, exist_ok=True)

    count = 0
    skipped = 0

    for img_path in class_dir.iterdir():
        if img_path.suffix.lower() not in valid_exts:
            skipped += 1
            continue

        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize((224, 224))
            img.save(out_class_dir / f"{class_dir.name}_{count}.jpg", "JPEG")
            count += 1
        except Exception:
            skipped += 1

    print(class_dir.name, "saved:", count, "| skipped:", skipped)

print("✅ Clean dataset ready:", CLEAN_DIR)

healthy saved: 30 | skipped: 0
leaf_spot saved: 16 | skipped: 0
mummy_berry saved: 21 | skipped: 0
botrytis saved: 26 | skipped: 0
anthracnose saved: 23 | skipped: 0
✅ Clean dataset ready: /content/blueberry_dataset_clean


In [4]:
import random
import shutil

ORIGINAL_DIR = CLEAN_DIR

SPLIT_DIR = Path("/content/blueberry_dataset_split")

TRAIN_DIR = SPLIT_DIR / "train"
VAL_DIR = SPLIT_DIR / "val"

if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

TRAIN_DIR.mkdir(parents=True)
VAL_DIR.mkdir(parents=True)

random.seed(42)

for class_dir in ORIGINAL_DIR.iterdir():

    if not class_dir.is_dir():
        continue

    images = [
        p for p in class_dir.iterdir()
        if p.suffix.lower() in valid_exts
    ]

    random.shuffle(images)

    split = int(len(images) * 0.8)

    train_images = images[:split]
    val_images = images[split:]

    (TRAIN_DIR / class_dir.name).mkdir()
    (VAL_DIR / class_dir.name).mkdir()

    for img in train_images:
        shutil.copy(img, TRAIN_DIR / class_dir.name / img.name)

    for img in val_images:
        shutil.copy(img, VAL_DIR / class_dir.name / img.name)

    print(class_dir.name)
    print("Train:", len(train_images))
    print("Validation:", len(val_images))
    print("----------------")

healthy
Train: 24
Validation: 6
----------------
leaf_spot
Train: 12
Validation: 4
----------------
mummy_berry
Train: 16
Validation: 5
----------------
botrytis
Train: 20
Validation: 6
----------------
anthracnose
Train: 18
Validation: 5
----------------


In [5]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 8

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print(class_names)

Found 90 files belonging to 5 classes.
Found 26 files belonging to 5 classes.
['anthracnose', 'botrytis', 'healthy', 'leaf_spot', 'mummy_berry']


In [6]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(100).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [7]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15),
])

In [8]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications.efficientnet import preprocess_input

num_classes = len(class_names)

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,055,976 (15.47 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [9]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from pathlib import Path

MODEL_DIR = Path("/content/drive/MyDrive/BlueberryDiseaseProject/model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = MODEL_DIR / "best_blueberry_model.keras"

callbacks = [
    EarlyStopping(
        monitor="val_accuracy",
        patience=6,
        restore_best_weights=True,
        mode="max"
    ),
    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6
    )
]

In [10]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=callbacks
)

Epoch 1/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 29s 1s/step - accuracy: 0.2111 - loss: 1.7453 - val_accuracy: 0.4231 - val_loss: 1.4516 - learning_rate: 0.0010
Epoch 2/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 765ms/step - accuracy: 0.4000 - loss: 1.4235 - val_accuracy: 0.5385 - val_loss: 1.3357 - learning_rate: 0.0010
Epoch 3/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 938ms/step - accuracy: 0.5000 - loss: 1.2382 - val_accuracy: 0.5769 - val_loss: 1.2531 - learning_rate: 0.0010
Epoch 4/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 730ms/step - accuracy: 0.5444 - loss: 1.1090 - val_accuracy: 0.5769 - val_loss: 1.1901 - learning_rate: 0.0010
Epoch 5/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 731ms/step - accuracy: 0.6667 - loss: 0.9982 - val_accuracy: 0.6154 - val_loss: 1.1359 - learning_rate: 0.0010
Epoch 6/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 10s 873ms/step - accuracy: 0.6667 - loss: 0.9180 - val_accuracy: 0.5769 - val_loss: 1.1007 - learning_rate: 0.0010
Epoch 7/25
12/12 ━━━━━━━━━━━━━━━━━━━━ 7s 621ms/step - accuracy: 0.7333 - loss: 0.8153 - 

In [11]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.4778 - loss: 1.3238 - val_accuracy: 0.6923 - val_loss: 1.0088 - learning_rate: 1.0000e-05
Epoch 2/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 10s 820ms/step - accuracy: 0.4889 - loss: 1.2402 - val_accuracy: 0.6538 - val_loss: 1.0019 - learning_rate: 1.0000e-05
Epoch 3/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 895ms/step - accuracy: 0.5111 - loss: 1.3107 - val_accuracy: 0.6538 - val_loss: 1.0005 - learning_rate: 1.0000e-05
Epoch 4/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 957ms/step - accuracy: 0.5444 - loss: 1.1807 - val_accuracy: 0.6154 - val_loss: 1.0045 - learning_rate: 1.0000e-05
Epoch 5/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 10s 908ms/step - accuracy: 0.5444 - loss: 1.1257 - val_accuracy: 0.6154 - val_loss: 1.0105 - learning_rate: 1.0000e-05
Epoch 6/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 10s 816ms/step - accuracy: 0.5333 - loss: 1.1326 - val_accuracy: 0.6154 - val_loss: 1.0212 - learning_rate: 1.0000e-05
Epoch 7/10
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 964ms/step - accura

In [12]:
import json

final_model_path = MODEL_DIR / "blueberry_disease_classifier.keras"
model.save(final_model_path)

class_names_path = MODEL_DIR / "class_names.json"
with open(class_names_path, "w") as f:
    json.dump(class_names, f)

print("✅ Model saved:", final_model_path)
print("✅ Class names saved:", class_names_path)

✅ Model saved: /content/drive/MyDrive/BlueberryDiseaseProject/model/blueberry_disease_classifier.keras
✅ Class names saved: /content/drive/MyDrive/BlueberryDiseaseProject/model/class_names.json


In [13]:
loss, accuracy = model.evaluate(val_ds)

print(f"Validation Accuracy: {accuracy*100:.2f}%")
print(f"Validation Loss: {loss:.4f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 363ms/step - accuracy: 0.6923 - loss: 1.0088
Validation Accuracy: 69.23%
Validation Loss: 1.0088


In [14]:
import numpy as np
from PIL import Image

def predict_image(image_path):
    img = Image.open(image_path).convert("RGB")
    img = img.resize((224,224))

    arr = np.array(img)
    arr = np.expand_dims(arr,0)

    pred = model.predict(arr, verbose=0)[0]

    idx = np.argmax(pred)

    print("Prediction:", class_names[idx])
    print("Confidence:", round(float(pred[idx])*100,2),"%")

In [18]:
!pip install huggingface_hub -q

In [16]:
from huggingface_hub import notebook_login

notebook_login()

In [17]:
from huggingface_hub import HfApi
from pathlib import Path

api = HfApi()

repo_id = "MaayanSal20/blueberry-disease-classifier"

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True
)

api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/BlueberryDiseaseProject/model/blueberry_disease_classifier.keras",
    path_in_repo="blueberry_disease_classifier.keras",
    repo_id=repo_id,
    repo_type="model"
)

api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/BlueberryDiseaseProject/model/class_names.json",
    path_in_repo="class_names.json",
    repo_id=repo_id,
    repo_type="model"
)

print("Uploaded to:", repo_id)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._disease_classifier.keras:  27%|##7       | 7.91MB / 29.1MB            

Uploaded to: MaayanSal20/blueberry-disease-classifier
